In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Disable auto-scroll in notebook output for smooth interaction
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def textbook_oversampling_spectrum_simulation(L):
    clear_output(wait=True)
    
    # High-resolution frequency axis covering the entire baseband and beyond
    omega = np.linspace(-3 * np.pi, 3.0 * np.pi, 2000)
    
    # Strict bandwidth setting matching the textbook example: omega_N = 0.2 * pi
    omega_N = 0.2 * np.pi  
    
    # Original spectrum function for Plot 1
    def original_triangle_spect(w, scale=1.0):
        spec = np.zeros_like(w)
        mask = np.abs(w) <= omega_N
        spec[mask] = scale * (1.0 - np.abs(w[mask]) / omega_N)
        return spec

    # Expanded/Compressed replica spectrum function for Plot 2 & 3: X_e(e^{j\omega}) = X(e^{j\omega L})
    def expanded_triangle_spect(w, factor_L, scale=1.0):
        spec = np.zeros_like(w)
        effective_omega_N = omega_N / factor_L  # Σωστή συμπίεση εύρους ζώνης κατά L
        mask = np.abs(w) <= effective_omega_N
        spec[mask] = scale * (1.0 - np.abs(w[mask]) / effective_omega_N)
        return spec

    # Δημιουργία φιγούρας με σταθερή κλίμακα άξονα y (έως 10, για να χωράει άνετα έως L=8)
    fig, axes = plt.subplots(3, 1, figsize=(12, 8))
    y_fixed_limit = 10.0 
    
    # --- PLOT 1: Original Spectrum X(e^{j\omega}) ---
    spec_orig = original_triangle_spect(omega, scale=1.0)
    axes[0].plot(omega, spec_orig, 'tab:blue', linewidth=2, label=r'Original Spectrum $\mathcal{X}(e^{j\omega})$')
    axes[0].fill_between(omega, 0, spec_orig, where=(spec_orig > 1e-5), color='tab:blue', alpha=0.2)
    axes[0].axvline(x=omega_N, color='orange', linestyle='--', label=r'Bandwidth $\Omega_N = 0.2\pi$')
    axes[0].axvline(x=-omega_N, color='orange', linestyle='--')
    axes[0].set_title(r'1. Original Continuous Spectrum $\mathcal{X}(e^{j\omega})$', fontsize=9, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=9)
    axes[0].set_xlim(-2*np.pi, 2*np.pi)
    axes[0].set_ylim(-0.05, y_fixed_limit)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(loc='upper right', frameon=True, fontsize=8)

    # --- PLOT 2: Expanded Spectrum X_e(e^{j\omega}) = X(e^{j\omega L}) ---
    spec_expanded = np.zeros_like(omega)
    for k in range(-6, 7):
        replica_center = k * (2.0 * np.pi / L)
        spec_expanded += expanded_triangle_spect(omega - replica_center, L, scale=1.0)

    axes[1].plot(omega, spec_expanded, 'tab:purple', linewidth=2, label=r'Expanded Spectrum $\mathcal{X}_e(e^{j\omega}) = \mathcal{X}(e^{j\omega L})$')
    axes[1].fill_between(omega, 0, spec_expanded, where=(spec_expanded > 1e-5), color='tab:purple', alpha=0.2)
    axes[1].axvline(x=np.pi/L, color='green', linestyle=':', linewidth=2.0, label=fr'Filter Cutoff $\pi/L$ ($\pi/{L}$)')
    axes[1].axvline(x=-np.pi/L, color='green', linestyle=':', linewidth=2.0)
    axes[1].set_title(fr'2. Expanded Spectrum after Expander (Zero Insertion, $L = {L}$)', fontsize=9, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=9)
    axes[1].set_xlim(-2*np.pi, 2*np.pi)
    axes[1].set_ylim(-0.05, y_fixed_limit)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(loc='upper right', frameon=True, fontsize=8)

    # --- PLOT 3: Final Interpolated / Oversampled Spectrum ---
    H_L = np.zeros_like(omega)
    filter_mask = np.abs(omega) <= (np.pi / L)
    H_L[filter_mask] = L
    
    interpolated_output = spec_expanded * H_L

    axes[2].plot(omega, interpolated_output, color='tab:red', linewidth=2.5, label=r'Final Interpolated Spectrum $\mathcal{X}_u(e^{j\omega})$')
    axes[2].fill_between(omega, 0, interpolated_output, where=(interpolated_output > 1e-5), color='tab:red', alpha=0.3)

    axes[2].axvline(x=np.pi/L, color='green', linestyle=':', linewidth=2.0, label=fr'New Cutoff Limit $\pi/L$ ($\pi/{L}$)')
    axes[2].axvline(x=-np.pi/L, color='green', linestyle=':', linewidth=2.0)
    
    axes[2].set_title(fr'3. Final Interpolator Output Spectrum ($L = {L}$, Filter Gain = ${L}$)', fontsize=9, fontweight='bold', color='black')
    axes[2].set_xlabel(r'Digital Frequency ($\omega$)', fontsize=9)
    axes[2].set_ylabel('Amplitude', fontsize=9)
    axes[2].set_xlim(-2*np.pi, 2*np.pi)
    axes[2].set_ylim(-0.05, y_fixed_limit)
    axes[2].set_xticks([-2*np.pi, -np.pi, -np.pi/L, 0, np.pi/L, np.pi, 2*np.pi])
    axes[2].set_xticklabels([r'$-2\pi$', r'$-\pi$', r'$-\pi/L$', r'$0$', r'$\pi/L$', r'$\pi$', r'$2\pi$'])
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].legend(loc='upper right', frameon=True, fontsize=8)

    plt.tight_layout()
    plt.show()
    
    # Print dynamic parameters
    print(f"--- Oversampling / Interpolation Parameters for L = {L} ---")
    print(f"• Filter Amplitude Gain (Height): L = {L}")
    print(f"• Spectral Replica Spacing (Shift): 2*pi / L = 2*pi / {L}")
    print(f"• New Digital Bandwidth Limit: pi / L = pi / {L} ≈ {np.pi/L:.3f} rad/sample")
    print(f"• Scaled Replica Half-Bandwidth: 0.2*pi / {L} ≈ {0.2*np.pi/L:.3f} rad/sample")
    print(f"Status: Interpolation successful (Filter cleanly isolates the central replica scaled by {L}).")

# Interactive widget slider for oversampling factor L up to 8
l_slider = widgets.IntSlider(
    value=3, min=1, max=8, step=1,
    description='Oversampling Factor ($L$):',
    style={'description_width': 'initial'}
)

ui_l = widgets.VBox([l_slider])
display(ui_l)

out_l = widgets.interactive_output(textbook_oversampling_spectrum_simulation, {'L': l_slider})
display(out_l)